In [2]:
pip install matplotlib

   ---------------------------------------- 0.0/9.3 MB ? eta -:--:--
   --------------- ------------------------ 3.7/9.3 MB 18.1 MB/s eta 0:00:01
   ------------------------------------- -- 8.7/9.3 MB 20.9 MB/s eta 0:00:01
   ---------------------------------------- 9.3/9.3 MB 19.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ---------------------------------------- 2.3/2.3 MB 20.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/7.2 MB ? eta -:--:--
   ----------------------- ---------------- 4.2/7.2 MB 21.2 MB/s eta 0:00:01
   ---------------------------------------- 7.2/7.2 MB 19.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt

In [4]:
prices = pd.read_csv(
    "../data/raw/sp500_stocks.csv",
    index_col=0,
    parse_dates=True
)

market = pd.read_csv(
    "../data/raw/sp500_index.csv",
    index_col=0,
    parse_dates=True
)

rf = pd.read_csv(
    "../data/raw/risk_free.csv",
    index_col=0,
    parse_dates=True
)

In [20]:
constituents = pd.read_csv(
    "https://raw.githubusercontent.com/datasets/s-and-p-500-companies/master/data/constituents.csv"
)

In [21]:
sector_map = dict(
    zip(
        constituents["Symbol"],
        constituents["GICS Sector"]
    )
)

In [5]:
stock_returns = prices.pct_change().dropna()

market_returns = market.pct_change().dropna()
market_returns.columns = ["Market"]

rf = rf / 100

daily_rf = (1 + rf) ** (1/252) - 1
daily_rf.columns = ["RF"]

In [9]:
stock = "AAPL"

data = pd.concat(
    [
        stock_returns[stock],
        market_returns["Market"],
        daily_rf["RF"]
    ],
    axis=1
).dropna()

data.columns = ["Stock", "Market", "RF"]

stock_excess = data["Stock"] - data["RF"]
market_excess = data["Market"] - data["RF"]

C:\Users\kotha\AppData\Local\Temp\ipykernel_13160\1986275703.py:3: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  data = pd.concat(


In [12]:
market_excess.name = "Market"
X = sm.add_constant(market_excess)

y = stock_excess

model = sm.OLS(y, X).fit()

In [13]:
print(model.params)
print(model.rsquared)

const     0.000451
Market    1.203310
dtype: float64
0.5580301308304114


In [31]:
results = []

In [32]:
for stock in stock_returns.columns:

    data = pd.concat(
        [
            stock_returns[stock],
            market_returns["Market"],
            daily_rf["RF"]
        ],
        axis=1
    ).dropna()

    data.columns = ["Stock", "Market", "RF"]

    stock_excess = data["Stock"] - data["RF"]
    market_excess = data["Market"] - data["RF"]

    market_excess.name = "Market"

    X = sm.add_constant(market_excess)
    y = stock_excess

    model = sm.OLS(y, X).fit()

    results.append({
        "Stock": stock,
        "Alpha": model.params["const"],
        "Beta": model.params["Market"],
        "Alpha_p": model.pvalues["const"],
        "Beta_p": model.pvalues["Market"],
        "R2": model.rsquared,
        "Sector": sector_map.get(stock, "Unknown")
    })

C:\Users\kotha\AppData\Local\Temp\ipykernel_13160\674177908.py:3: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  data = pd.concat(
C:\Users\kotha\AppData\Local\Temp\ipykernel_13160\674177908.py:3: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  data = pd.concat(
C:\Users\kotha\AppData\Local\Temp\ipykernel_13160\674177908.py:3: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True

In [33]:
results_df = pd.DataFrame(results)
results_df.head()

,Stock,Alpha,Beta,Alpha_p,Beta_p,R2,Sector
0,AAPL,0.000451,1.203310,0.050055,0.000000e+00,0.558030,Information Technology
1,ABBV,0.000407,0.653133,0.155796,3.094928e-131,0.193653,Health Care
2,BAC,0.000083,1.209051,0.750442,0.000000e+00,0.495953,Financials
3,CAT,0.000415,1.072111,0.133030,0.000000e+00,0.410417,Industrials
4,COP,-0.000017,1.118597,0.965281,1.338724e-197,0.278052,Energy


In [27]:
print("AAPL" in sector_map)
print(sector_map.get("AAPL"))

True
Information Technology


In [34]:
results_df.sort_values("Beta", ascending=False)

,Stock,Alpha,Beta,Alpha_p,Beta_p,R2,Sector
20,NVDA,0.001693,1.758900,0.000092,0.000000e+00,0.433020,Information Technology
2,BAC,0.000083,1.209051,0.750442,0.000000e+00,0.495953,Financials
0,AAPL,0.000451,1.203310,0.050055,0.000000e+00,0.558030,Information Technology
10,GS,0.000201,1.203233,0.391133,0.000000e+00,0.550155,Financials
19,MSFT,0.000467,1.185112,0.018439,0.000000e+00,0.623037,Information Technology
9,GOOGL,0.000504,1.141315,0.038910,0.000000e+00,0.502263,Communication Services
16,MA,0.000296,1.126712,0.145499,0.000000e+00,0.585907,Financials
25,SLB,-0.000420,1.120558,0.295978,3.617057e-186,0.264167,Energy
4,COP,-0.000017,1.118597,0.965281,1.338724e-197,0.278052,Energy
8,GE,0.000091,1.103151,0.790001,2.095038e-238,0.325515,Industrials


In [35]:
results_df.to_csv(
    "../data/processed/capm_results.csv",
    index=False
)